In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, GenerationConfig


d:\desktop\Graduation Project\GRAD_PROJECT\invoice-ai-project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
import pandas as pd

In [4]:
model_save_path = r"D:\desktop\\Graduation Project\\GRAD_PROJECT\\invoice-ai-project\\ai_core\\models\\Training_model"

instruct_model = AutoModelForSeq2SeqLM.from_pretrained(model_save_path)
instruct_tokenizer = AutoTokenizer.from_pretrained(model_save_path)

In [5]:
words_img = pd.read_csv("testing_image.csv")

In [6]:
words_img.head()

,top_left_x,top_right_x,bottom_right_x,bottom_left_x,top_left_y,top_right_y,bottom_right_y,bottom_left_y,text,confidence
0,91,727,727,91,209,209,598,598,الأمإن,0.367153
1,3879,4782,4782,3879,619,619,759,759,هف:شداد ٣,0.015202
2,65,769,769,65,684,684,856,856,٥٧٥٨٤٤٥,0.315640
3,78,768,768,78,848,848,984,984,لحلول الأعسمال,0.298145
4,83,772,772,83,960,960,1071,1071,٥٨5ز٥٧؟ 55زء,0.003868


In [7]:
text = words_img.text
text

0             الأمإن
1          هف:شداد ٣
2            ٥٧٥٨٤٤٥
3     لحلول الأعسمال
4      ٥٨5ز٥٧؟ 55زء 
           ...      
63              لاصق
64             أحبار
65               عبو
66           المجموع
67           المجموع
Name: text, Length: 68, dtype: object

In [23]:
instruct_model_corrections = []

for distorted_text in text:
    prompt = f"""
صحح الكلمة أو الجملة التالية:

{distorted_text}

الجملة المصححة:
"""
    input_ids = instruct_tokenizer(prompt, return_tensors="pt").input_ids

    

    instruct_model_outputs = instruct_model.generate(
        input_ids=input_ids,
        generation_config=GenerationConfig(max_new_tokens=200)
    )
    instruct_model_text_output = instruct_tokenizer.decode(instruct_model_outputs[0], skip_special_tokens=True)
    instruct_model_corrections.append(instruct_model_text_output)

results_df = pd.DataFrame({
    'distorted_text': text,
    'instruct_model_correction': instruct_model_corrections
})


print(results_df)


    distorted_text                          instruct_model_correction
0           الأمإن  صحح الكلمة أو الجملة التالية : الأم إن الجملة ...
1        هف:شداد ٣  صحح الكلمة أو الجملة التالية : هف : شداد ، ٣ ا...
2          ٥٧٥٨٤٤٥  صحح الكلمة أو الجملة التالية : ٥٧٥٨٤٤٥ الجملة ...
3   لحلول الأعسمال  صحح الكلمة أو الجملة التالية : لحلول الأعسمال ...
4    ٥٨5ز٥٧؟ 55زء   صحح الكلمة أو الجملة التالية : ٥٨5 ، ز٥٧ ؟ 55 ...
..             ...                                                ...
63            لاصق  صحح الكلمة أو الجملة التالية : لاصق الجملة الم...
64           أحبار  صحح الكلمة أو الجملة التالية : أحبار الجملة ال...
65             عبو  صحح الكلمة أو الجملة التالية : عبو الجملة المص...
66         المجموع  صحح الكلمة أو الجملة التالية : المجموع الجملة ...
67         المجموع  صحح الكلمة أو الجملة التالية : المجموع الجملة ...

[68 rows x 2 columns]


In [24]:
results_df.to_csv("test.csv",index=False, encoding="utf-8-sig")

In [12]:
import os
import re
import json
import Levenshtein
import pandas as pd

VOCAB_PATH = 'vocabs.json'

def load_vocab_from_json(file_path):
    if os.path.exists(file_path):
        with open(file_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
            vocab = data
            print(f"Loaded vocab from JSON. Number of words: {len(vocab)}")
            return set(vocab)
    else:
        print("Vocab file not found.")
        return set()

jamid_names = load_vocab_from_json(VOCAB_PATH)

if jamid_names:
    print("Sample of vocab words:")
    for name in list(jamid_names)[:20]:
        print(name)
else:
    print("No vocab words loaded.")

def preprocess_arabic_word(word):
    word = re.sub(r'[\u064B-\u065F]', '', word)
    word = word.replace('أ', 'ا').replace('إ', 'ا').replace('آ', 'ا')
    return word

def correct_word(word, correct_words):
    word = preprocess_arabic_word(word)
    if word in correct_words:
        return word

    suggestions = []
    for correct_word in correct_words:
        clean_correct_word = preprocess_arabic_word(correct_word)
        distance = Levenshtein.distance(word, clean_correct_word)
        if distance <= 3:
            suggestions.append((correct_word, distance))

    if suggestions:
        suggestions.sort(key=lambda x: x[1])
        return suggestions[0][0]
    else:
        return word


Loaded vocab from JSON. Number of words: 185
Sample of vocab words:
الدفع
أعطال
التوصيل
الكلي
شبكة
تقنية
خصم
تحويل
أسبوعية
استضافة
دوري
داخلية
الأعمال
شهري
البيع
مشتريات
قلم
موقع
على
أوراق


In [17]:
import pandas as pd

# دالة لتصحيح كل كلمة داخل النص
def correct_text_column(text, correct_words):
    if pd.isna(text):
        return text

    words = re.findall(r'[\u0600-\u06FF]+', text)
    corrected_words = [correct_word(word, correct_words) for word in words]
    return " ".join(corrected_words)

# تطبيق التصحيح على العمود
words_img['corrected_text'] = words_img['text'].apply(lambda x: correct_text_column(x, jamid_names))

# حفظ النتائج في ملف جديد
words_img.to_csv("corrected_text_output.csv", index=False, encoding='utf-8-sig')

# عرض عينة من النتائج
print(words_img[['text', 'corrected_text']].head(10))


             text corrected_text
0          الأمإن         الضمان
1       هف:شداد ٣  هاتف شراء خصم
2         ٥٧٥٨٤٤٥        ٥٧٥٨٤٤٥
3  لحلول الأعسمال  محمول الأعمال
4   ٥٨5ز٥٧؟ 55زء    خصم ز٥٧؟ ماء
5     أحمد مهايني      اسم معدني
6               1               
7  رقم الفاتورة :   رقم الفاتورة
8       1/18/2024               
9             حدة            مدة
